# Final Ensemble — Fine-Tuned Transfer Learning Models
## Computer Vision Project — UC3M — CodaBench Submission

This notebook produces the **final CodaBench prediction file (`output_ft.csv`)** for the fine-tuned model ensemble.

### Overview
This is the production ensemble notebook. It loads all trained transfer learning base models, generates their predictions on the full validation and test sets, trains a Logistic Regression meta-learner on validation predictions, and produces the final test-set probabilities for submission.

### Difference from `base_models_notebooks_ft/ensemble.ipynb`
The FT notebook in `base_models_notebooks_ft/` is the **development notebook** used to test and refine the ensemble approach. This notebook in `predictions/` is the **production version** — it runs the final configuration of base models and saves `output_ft.csv` for CodaBench submission.

### Base Models (Final Configuration)
| Model | Val AUC | Notes |
|-------|---------|-------|
| ResNet50 | 0.8346 | Full fine-tuning |
| EfficientNet-B4 | ~0.82 | Last 3 blocks |
| Swin-B | 0.7871 | Last transformer stage |
| ConvNeXt-Base | 0.8031 | Last feature stage |
| ViT-B/16 | 0.7539 | Last encoder layer |
| EfficientNet-B5 | 0.7345 | Last feature stage |

**Meta-learner Val AUC: 0.8476** — the ensemble consistently outperforms every individual base model.

## 1. Imports and Configuration

In [ ]:
import os
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torchvision import transforms
from torchvision.models import vgg16
!pip install fastai
from fastai.vision.all import xresnet50
from skimage import io, transform, color
import cv2
from torch.utils.data import random_split
from torchvision.models import efficientnet_b5
from sklearn.metrics import roc_auc_score

import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Dataset and Preprocessing

The same preprocessing pipeline used during individual model training is replicated exactly. Consistent preprocessing is **mandatory** — each model receives the same image representation it was fine-tuned on, ensuring its learned features activate correctly.

### Dataset Class

Standard `RetinopathyDataset` — right-eye mirroring and binary labels.

In [ ]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, dtype={'id': str, 'eye': int, 'label': int})

        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(len(self.dataset))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)

        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, 'images')
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset.iloc[idx]
        img_path = os.path.join(self.img_dir, row.id + '.jpg')
        image = io.imread(img_path)

        if row.eye == 1:
            image = image[:, ::-1, :]

        label = int(row.label > 0)

        sample = {'image': image, 'label': label}

        if self.transform:
            sample = self.transform(sample)

        return sample

### Preprocessing Transforms

`CropByEye`, `Rescale`, `CenterCrop`, `ToTensor`, `Normalize` — identical to individual model notebooks.

In [ ]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]

        gray = color.rgb2gray(image)
        _, mask = cv2.threshold(gray, self.threshold, 1, cv2.THRESH_BINARY)

        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return sample

        minx = max(sidx[1].min() - self.border[1], 0)
        maxx = min(sidx[1].max() + self.border[1], w)
        miny = max(sidx[0].min() - self.border[0], 0)
        maxy = min(sidx[0].max() + self.border[0], h)

        image = image[miny:maxy, minx:maxx]

        return {'image': image, 'label': label}


class Rescale(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        image = transform.resize(image, (self.size, self.size))
        return {'image': image, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        image = image.transpose((2, 0, 1))
        return {
            'image': torch.tensor(image, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.float32)
        }

### Augmentation Helpers

Defined for interface completeness. **No augmentation during inference.**

In [ ]:
class ToPIL(object):
    def __call__(self, sample):
        from PIL import Image
        image, label = sample['image'], sample['label']
        return {'image': Image.fromarray((image * 255).astype(np.uint8)), 'label': label}


class FromPIL(object):
    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        return {'image': np.array(image) / 255.0, 'label': label}

In [ ]:
class TVAugment(object):
    def __init__(self):
        from torchvision import transforms

        self.augment = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2)
        ])

    def __call__(self, sample):
        from PIL import Image

        image, label = sample['image'], sample['label']

        if image.max() <= 1:
            image = (image * 255).astype(np.uint8)
        else:
            image = image.astype(np.uint8)

        pil = Image.fromarray(image)
        pil = self.augment(pil)

        image = np.array(pil).astype(np.float32) / 255.0

        return {'image': image, 'label': label}

In [ ]:
class CenterCrop(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.size, self.size

        top = max((h - new_h) // 2, 0)
        left = max((w - new_w) // 2, 0)

        image = image[top:top+new_h, left:left+new_w]

        return {'image': image, 'label': label}

In [ ]:
class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std = np.array(std)

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        c, h, w = image.shape

        mean = torch.tensor(self.mean, dtype=image.dtype)
        std = torch.tensor(self.std, dtype=image.dtype)

        image = (image - mean[:, None, None]) / std[:, None, None]

        return {'image': image, 'label': label}

In [ ]:
class Rescale(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]

        if h > w:
            new_h = int(self.size * h / w)
            new_w = self.size
        else:
            new_h = self.size
            new_w = int(self.size * w / h)

        image = transform.resize(image, (new_h, new_w))

        return {'image': image, 'label': label}

### Transform Pipelines

Only the deterministic `val_transform` is used — no training-time randomness.

In [ ]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    Rescale(256),
    TVAugment(),
    CenterCrop(224),
    ToTensor(),
    Normalize(pixel_mean, pixel_std)
])

val_transform = transforms.Compose([
    CropByEye(0.10, 1),
    Rescale(256),
    CenterCrop(224),
    ToTensor(),
    Normalize(pixel_mean, pixel_std)
])

## 3. Data Loading

Full validation (500) and test (1000) sets are loaded. No shuffling to maintain consistent prediction ordering.

In [ ]:
data_dir = "/content/drive/MyDrive/CV_b/Project_2/data"

train_dataset = RetinopathyDataset(
    os.path.join(data_dir, "train.csv"),
    data_dir,
    transform=train_transform
)

val_dataset = RetinopathyDataset(
    os.path.join(data_dir, "val.csv"),
    data_dir,
    transform=val_transform
)

test_dataset = RetinopathyDataset(
    os.path.join(data_dir, "test.csv"),
    data_dir,
    transform=val_transform
)


## 4. Validation Set Split — Meta-Train / Meta-Val

The 500-image validation set is split 50/50:
- **meta-train (250 images):** base model predictions on this subset are used as features to train the Logistic Regression meta-learner
- **meta-val (250 images):** held out for unbiased AUC estimation of the ensemble

This split-within-validation ensures the meta-learner is never evaluated on data it was trained on. Using the full validation set (without splitting) would produce optimistic, biased AUC estimates.

In [ ]:
val_size = len(val_dataset)
meta_train_size = val_size // 2
meta_val_size = val_size - meta_train_size

meta_train_dataset, meta_val_dataset = random_split(
    val_dataset, [meta_train_size, meta_val_size]
)

meta_train_loader = DataLoader(meta_train_dataset, batch_size=64, shuffle=False)
meta_val_loader = DataLoader(meta_val_dataset, batch_size=64, shuffle=False)

## 5. Base Model Instantiation

All base model architectures are instantiated with `weights=None`. Checkpoint weights will be loaded in the next step. This approach cleanly separates architecture definition from weight loading.

In [ ]:
import torch
import torch.nn as nn
import os

from torchvision.models import resnet50
from torchvision.models import efficientnet_b4
from torchvision.models import swin_b
from torchvision.models import densenet121
from torchvision.models import convnext_tiny
from torchvision.models import vgg16_bn
from torchvision.models import vit_b_16
from torchvision.models import efficientnet_b5
# -------- ResNet50 --------
resnet = resnet50(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 1)

# -------- EfficientNet B4 --------
effnet = efficientnet_b4(weights=None)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 1)

# -------- Swin Transformer --------
swin = swin_b(weights=None)
swin.head = nn.Linear(swin.head.in_features, 1)

# -------- DenseNet121 --------
densenet = densenet121(weights=None)
densenet.classifier = nn.Linear(densenet.classifier.in_features, 1)

# -------- ConvNeXt Tiny --------
convnext = convnext_tiny(weights=None)
convnext.classifier[2] = nn.Linear(convnext.classifier[2].in_features, 1)

# -------- ViT --------
vit = vit_b_16(weights=None)
vit.heads.head = nn.Linear(vit.heads.head.in_features, 1)
vit = vit.to(device)

# -------- EfficientNet B5 --------
effnet_b5 = efficientnet_b5(weights=None)
effnet_b5.classifier[1] = nn.Linear(effnet_b5.classifier[1].in_features, 1)
effnet_b5 = effnet_b5.to(device)

# -------- VGG --------
#vgg = vgg16_bn(weights=None)
#vgg.classifier[6] = nn.Linear(vgg.classifier[6].in_features, 1)
#vgg = vgg.to(device)

# -------- xresnet --------
#xresnet = xresnet50(pretrained=False)

# cambiar última capa correctamente
#xresnet[11] = nn.Linear(xresnet[11].in_features, 1)

#xresnet = xresnet.to(device)


# Move to device
resnet = resnet.to(device)
effnet = effnet.to(device)
swin = swin.to(device)
densenet = densenet.to(device)
convnext = convnext.to(device)

# Load weights
models_dir = "/content/drive/MyDrive/CV/Project2/Models"



## 6. Loading Saved Checkpoints

Best-epoch model weights (saved during individual training) are loaded from Google Drive. For large models (DenseNet121, ConvNeXt-Tiny) stored as zip files, they are decompressed first.

In [ ]:
models_dir = "/content/drive/MyDrive/CV_b/Project_2/Models"

resnet.load_state_dict(torch.load(os.path.join(models_dir, "best_model_resnet50.pth")))
effnet.load_state_dict(torch.load(os.path.join(models_dir, "best_model_effnet.pth")))
swin.load_state_dict(torch.load(os.path.join(models_dir, "best_model_swin.pth")))
vit.load_state_dict(torch.load(os.path.join(models_dir, "best_model_vit.pth")))
effnet_b5.load_state_dict(torch.load(os.path.join(models_dir, "best_model_eff5.pth")))
#vgg.load_state_dict(torch.load(os.path.join(models_dir, "best_model_vgg.pth")))
#xresnet.load_state_dict(torch.load(os.path.join(models_dir, "best_model_xresnet.pth")))

<All keys matched successfully>

In [ ]:
import zipfile
import os

models_dir = "/content/drive/MyDrive/CV_b/Project_2/Models"

# Archivos zip
zip_densenet = os.path.join(models_dir, "best_model_densenet.pth.zip")
zip_convnext = os.path.join(models_dir, "best_model_convextiny.pth.zip")
# Descomprimir DenseNet
with zipfile.ZipFile(zip_densenet, 'r') as zip_ref:
    zip_ref.extractall(models_dir)

# Descomprimir ConvNeXt
with zipfile.ZipFile(zip_convnext, 'r') as zip_ref:
    zip_ref.extractall(models_dir)

print("Unzipped correctly!")

Unzipped correctly!


In [ ]:
import os

models_dir = "/content/drive/MyDrive/CV_b/Project_2/Models"

print(os.listdir(models_dir))

['best_model_eff5.pth', 'best_model_swin.pth', 'best_model_effnet.pth', 'best_model_custom_git.pth', 'best_model_convextiny.pth.zip', 'best_model_densenet.pth.zip', 'best_model_resnet50.pth', 'best_model_resnet50', 'best_model_vgg.pth', 'best_model_xresnet.pth', 'best_customnet2.pth', 'best_customnet3.pth', 'best_customnet5.pth', 'best_customnet4.pth', 'best_model_vit.pth']


In [ ]:
densenet_path = None
convnext_path = None

for root, dirs, files in os.walk(models_dir):
    for file in files:
        if "densenet" in file:
            densenet_path = os.path.join(root, file)
        if "conv" in file:
            convnext_path = os.path.join(root, file)

print(densenet_path, convnext_path)

densenet.load_state_dict(torch.load(densenet_path))
convnext.load_state_dict(torch.load(convnext_path))

/content/drive/MyDrive/CV_b/Project_2/Models/best_model_densenet.pth.zip /content/drive/MyDrive/CV_b/Project_2/Models/best_model_convextiny.pth.zip


<All keys matched successfully>

## 7. Base Model Inference Function

The `predict` function runs each model in eval mode with `torch.no_grad()` for memory-efficient inference, returning sigmoid probabilities reshaped as (N, 1) column vectors — ready for feature matrix concatenation.

In [ ]:
def predict(model, loader):
    model.eval()
    outputs = []

    with torch.no_grad():
        for batch in loader:
            inputs = batch['image'].to(device)
            preds = torch.sigmoid(model(inputs).squeeze())
            outputs.extend(preds.cpu().numpy())

    return np.array(outputs).reshape(-1, 1)

## 8. Meta-Train Feature Matrix

Base model predictions on the 250-image meta-train split are concatenated into a feature matrix `X_meta_train` of shape (250, num_models). The corresponding binary labels `y_meta_train` are extracted directly from the dataset.

In [ ]:
# META TRAIN
res_pred_train = predict(resnet, meta_train_loader)
eff_pred_train = predict(effnet, meta_train_loader)
swin_pred_train = predict(swin, meta_train_loader)
densenet_pred_train = predict(densenet, meta_train_loader)
convnext_pred_train = predict(convnext, meta_train_loader)
vit_pred_train = predict(vit, meta_train_loader)
effb5_pred_train = predict(effnet_b5, meta_train_loader)
#vgg_pred_train = predict(vgg, meta_train_loader)
#xresnet_pred_train = predict(xresnet, meta_train_loader)
X_meta_train = np.concatenate([
    res_pred_train,
    eff_pred_train,
    swin_pred_train,
    densenet_pred_train,
    convnext_pred_train,
    vit_pred_train,
    effb5_pred_train,
    #vgg_pred_train,
    #xresnet_pred_train
], axis=1)
y_meta_train = np.array([
    meta_train_dataset[i]['label'] for i in range(len(meta_train_dataset))
])

## 9. Meta-Val Feature Matrix

Same process applied to the 250-image meta-val split → `X_meta_val` and `y_meta_val` for ensemble evaluation.

In [ ]:
res_pred_val = predict(resnet, meta_val_loader)
eff_pred_val = predict(effnet, meta_val_loader)
swin_pred_val = predict(swin, meta_val_loader)
densenet_pred_val = predict(densenet, meta_val_loader)
convnext_pred_val = predict(convnext, meta_val_loader)
vit_pred_val = predict(vit, meta_val_loader)
effb5_pred_val = predict(effnet_b5, meta_val_loader)
#vgg_pred_val = predict(vgg, meta_val_loader)
#xresnet_pred_val = predict(xresnet, meta_val_loader)
X_meta_val = np.concatenate([
    res_pred_val,
    eff_pred_val,
    swin_pred_val,
    densenet_pred_val,
    convnext_pred_val,
    vit_pred_val,
    effb5_pred_val,
    #vgg_pred_val,
    #xresnet_pred_val
], axis=1)
y_meta_val = np.array([
    meta_val_dataset[i]['label'] for i in range(len(meta_val_dataset))
])

## 10. Logistic Regression Meta-Learner

A `LogisticRegression` is trained on the 250×(num_models) meta-train feature matrix. It learns optimal weights for each base model's contribution to the ensemble prediction.

**Why Logistic Regression as meta-learner?**
- Simple, interpretable, and low-risk of overfitting on 250 samples
- The learned coefficients reveal each model's relative contribution
- Produces calibrated probabilities, important for the AUC metric
- Fast to train — enables rapid experimentation with different model subsets

After training, the meta-learner AUC on meta-val is reported as the ensemble's validation performance.

In [ ]:
from sklearn.linear_model import LogisticRegression
meta_model = LogisticRegression()

meta_model.fit(X_meta_train, y_meta_train)

LogisticRegression()

In [ ]:
val_preds = meta_model.predict_proba(X_meta_val)[:, 1]

auc = roc_auc_score(y_meta_val, val_preds)
print("Meta-model AUC:", auc)

Meta-model AUC: 0.8478170478170478


## 11. Final Test Set Predictions

Base model predictions on the 1000-image test set are collected, concatenated into `X_test_meta`, and fed to the trained meta-learner. The output (`final_preds`) is the ensemble's probability estimate for each test image. These are saved as `ensemble_submission2.csv` (renamed to `output_ft.csv` for CodaBench submission in `generate_submission.ipynb`).

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
res_pred_test = predict(resnet, test_loader)
eff_pred_test = predict(effnet, test_loader)
swin_pred_test = predict(swin, test_loader)
vit_pred_test = predict(vit, test_loader)
#vgg_pred_test = predict(vgg, test_loader)
#xresnet_pred_test = predict(xresnet, test_loader)
densenet_pred_test = predict(densenet, test_loader)
convnext_pred_test = predict(convnext, test_loader)
effb5_pred_test = predict(effnet_b5, test_loader)
X_test_meta = np.concatenate([
    res_pred_test,
    eff_pred_test,
    swin_pred_test,
    densenet_pred_test,
    convnext_pred_test,
    vit_pred_test,
    effb5_pred_test,
    #vgg_pred_test,
    #xresnet_pred_test
], axis=1)
final_preds = meta_model.predict_proba(X_test_meta)[:, 1]

In [ ]:
import pandas as pd

submission = pd.DataFrame({
    "prediction": final_preds
})

submission.to_csv("ensemble_submission2.csv", index=False)